# 09 — Natural transition orbital analysis

Inspect parsed ORCA natural-transition-orbital outputs, validate state mapping, summarize NTO pair weights, and prepare main-figure and Supporting Information cube-rendering manifests.


## A. Load and validate parsed NTO data


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("..").resolve()
RESULTS = ROOT / "results" / "excited_state" / "nto_multiwfn"

SYSTEM_ORDER = ["pdi", "pdi_terminal_functionalized"]
SYSTEM_LABELS = {
    "pdi": "Parent PDI",
    "pdi_terminal_functionalized": "Terminal-functionalized PDI",
}
MULTIPLICITY_ORDER = {"singlet": 0, "triplet": 1}

required_files = {
    "pairs": RESULTS / "nto_pairs.csv",
    "state_summary": RESULTS / "nto_state_summary.csv",
    "parse_summary": RESULTS / "nto_parse_summary.csv",
    "main_manifest": RESULTS / "nto_cube_manifest_main.csv",
    "si_manifest": RESULTS / "nto_cube_manifest_si.csv",
}

missing_files = [str(path) for path in required_files.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Run scripts.postprocess.parse_nto_calculations before this notebook. "
        f"Missing files: {missing_files}"
    )


In [2]:
pairs = pd.read_csv(required_files["pairs"])
state_summary = pd.read_csv(required_files["state_summary"])
parse_summary = pd.read_csv(required_files["parse_summary"])
main_manifest = pd.read_csv(required_files["main_manifest"])
si_manifest = pd.read_csv(required_files["si_manifest"])

required_pair_columns = {
    "system",
    "multiplicity",
    "state_label",
    "global_state_number",
    "nto_filename",
    "pair_index",
    "hole_orbital",
    "electron_orbital",
    "weight",
    "cumulative_weight",
}
required_state_columns = {
    "system",
    "multiplicity",
    "state_label",
    "excitation_energy_ev",
    "leading_pair_weight",
    "minimum_pairs_main",
    "recommended_pairs_main",
    "minimum_pairs_si",
    "recommended_pairs_si",
}

missing_pair_columns = required_pair_columns - set(pairs.columns)
missing_state_columns = required_state_columns - set(state_summary.columns)
if missing_pair_columns:
    raise ValueError(f"Missing columns in nto_pairs.csv: {sorted(missing_pair_columns)}")
if missing_state_columns:
    raise ValueError(f"Missing columns in nto_state_summary.csv: {sorted(missing_state_columns)}")

systems_present = set(state_summary["system"])
missing_systems = set(SYSTEM_ORDER) - systems_present
if missing_systems:
    print(f"Warning: no canonical NTO states parsed for {sorted(missing_systems)}")

for frame in [pairs, state_summary, main_manifest, si_manifest]:
    if "multiplicity" in frame.columns:
        frame["multiplicity_order"] = frame["multiplicity"].map(MULTIPLICITY_ORDER)
    if "system" in frame.columns:
        frame["system_order"] = frame["system"].map({system: i for i, system in enumerate(SYSTEM_ORDER)})


## B. Parse summary


In [3]:
display_columns = [
    "source_output",
    "inferred_system",
    "dedicated_multiplicity",
    "normal_termination",
    "number_of_singlet_states_parsed",
    "number_of_triplet_states_parsed",
    "number_of_nto_blocks_parsed",
    "number_of_matched_nto_blocks",
    "missing_nto_files",
    "energy_mismatches",
    "duplicate_conflict_status",
    "warnings",
]

display(parse_summary[display_columns])


,source_output,inferred_system,dedicated_multiplicity,normal_termination,number_of_singlet_states_parsed,number_of_triplet_states_parsed,number_of_nto_blocks_parsed,number_of_matched_nto_blocks,missing_nto_files,energy_mismatches,duplicate_conflict_status,warnings
0,calculations/pdi/multiwfn_analysis/excited_sta...,pdi,singlet,True,30,0,1,1,0,0,unique,NaN
1,calculations/pdi/multiwfn_analysis/excited_sta...,pdi,triplet,True,0,15,1,1,0,0,unique,NaN
2,calculations/pdi/multiwfn_analysis/excited_sta...,pdi,triplet,True,0,15,1,1,0,0,unique,NaN
3,calculations/pdi_terminal_functionalized/multi...,pdi_terminal_functionalized,singlet,True,30,0,1,1,0,0,unique,NaN
4,calculations/pdi_terminal_functionalized/multi...,pdi_terminal_functionalized,triplet,True,0,15,1,1,0,0,unique,NaN
5,calculations/pdi_terminal_functionalized/multi...,pdi_terminal_functionalized,triplet,True,0,15,1,1,0,0,unique,NaN


## C. Canonical NTO state summary


In [4]:
summary_display = (
    state_summary
    .sort_values(["system_order", "multiplicity_order", "local_state_index"])
    .drop(columns=["system_order", "multiplicity_order"], errors="ignore")
    .copy()
)

summary_display["excitation_energy_ev"] = summary_display["excitation_energy_ev"].round(3)
summary_display["leading_pair_weight"] = summary_display["leading_pair_weight"].round(5)
summary_display["total_printed_weight"] = summary_display["total_printed_weight"].round(5)
summary_display["cumulative_weight_main"] = summary_display["cumulative_weight_main"].round(5)
summary_display["cumulative_weight_si"] = summary_display["cumulative_weight_si"].round(5)

display(
    summary_display[
        [
            "system_label",
            "multiplicity_label",
            "state_label",
            "global_state_number",
            "excitation_energy_ev",
            "nto_filename",
            "nto_character",
            "leading_pair_weight",
            "total_printed_weight",
            "recommended_pairs_main",
            "recommended_pairs_si",
            "warnings",
        ]
    ]
)


,system_label,multiplicity_label,state_label,global_state_number,excitation_energy_ev,nto_filename,nto_character,leading_pair_weight,total_printed_weight,recommended_pairs_main,recommended_pairs_si,warnings
0,Parent PDI,Singlet,S1,1,2.763,pdi_S1_nto.mwfn,essentially single-pair,0.95548,0.99160,1,1,NaN
1,Parent PDI,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,essentially single-pair,0.90821,0.99852,1,1;2;3,NaN
2,Parent PDI,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,strongly mixed multi-pair state,0.49697,0.99681,1;2,1;2;3,NaN
3,Terminal-functionalized PDI,Singlet,S1,1,2.748,pdi_terminal_functionalized_S1_nto.mwfn,essentially single-pair,0.95515,0.99103,1,1,NaN
4,Terminal-functionalized PDI,Triplet,T1,1,1.605,pdi_terminal_functionalized_T1_nto.mwfn,essentially single-pair,0.90868,0.99846,1,1;2,NaN
5,Terminal-functionalized PDI,Triplet,T2,2,2.935,pdi_terminal_functionalized_T2_nto.mwfn,strongly mixed multi-pair state,0.49176,0.99668,1;2,1;2;3,NaN


## D. Cumulative NTO pair weights


In [5]:
pair_display = (
    pairs
    .sort_values(["system_order", "multiplicity_order", "local_state_index", "pair_index"])
    .drop(columns=["system_order", "multiplicity_order"], errors="ignore")
    .copy()
)

pair_display["weight"] = pair_display["weight"].round(6)
pair_display["cumulative_weight"] = pair_display["cumulative_weight"].round(6)

display(
    pair_display[
        [
            "system_label",
            "multiplicity_label",
            "state_label",
            "pair_index",
            "hole_orbital",
            "electron_orbital",
            "weight",
            "cumulative_weight",
            "parse_warning",
        ]
    ]
)


,system_label,multiplicity_label,state_label,pair_index,hole_orbital,electron_orbital,weight,cumulative_weight,parse_warning
0,Parent PDI,Singlet,S1,1,99a,100a,0.955480,0.955480,NaN
1,Parent PDI,Singlet,S1,2,98a,101a,0.019622,0.975102,NaN
2,Parent PDI,Singlet,S1,3,97a,102a,0.007424,0.982526,NaN
3,Parent PDI,Singlet,S1,4,96a,103a,0.003187,0.985713,NaN
4,Parent PDI,Singlet,S1,5,95a,104a,0.001965,0.987678,NaN
5,Parent PDI,Singlet,S1,6,94a,105a,0.001108,0.988786,NaN
6,Parent PDI,Singlet,S1,7,93a,106a,0.000918,0.989704,NaN
7,Parent PDI,Singlet,S1,8,92a,107a,0.000679,0.990383,NaN
8,Parent PDI,Singlet,S1,9,91a,108a,0.000613,0.990996,NaN
9,Parent PDI,Singlet,S1,10,90a,109a,0.000609,0.991605,NaN


## E. Main-figure rendering manifest


In [6]:
main_display = (
    main_manifest
    .sort_values(["system_order", "multiplicity_order", "state_label", "pair_index", "orbital_role"])
    .drop(columns=["system_order", "multiplicity_order"], errors="ignore")
    .copy()
)

main_display["excitation_energy_ev"] = main_display["excitation_energy_ev"].round(3)
main_display["pair_weight"] = main_display["pair_weight"].round(6)
main_display["cumulative_weight"] = main_display["cumulative_weight"].round(6)

display(main_display)


,system,system_label,multiplicity,multiplicity_label,state_label,global_state_number,excitation_energy_ev,nto_filename,nto_path,pair_index,pair_weight,cumulative_weight,orbital_role,orbital_label,recommended_cube_basename
1,pdi,Parent PDI,singlet,Singlet,S1,1,2.763,pdi_S1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.955480,0.955480,electron,100a,pdi_S1_nto_pair1_electron.cub
0,pdi,Parent PDI,singlet,Singlet,S1,1,2.763,pdi_S1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.955480,0.955480,hole,99a,pdi_S1_nto_pair1_hole.cub
3,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.908209,0.908209,electron,100a,pdi_T1_nto_pair1_electron.cub
2,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.908209,0.908209,hole,99a,pdi_T1_nto_pair1_hole.cub
5,pdi,Parent PDI,triplet,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.496971,0.496971,electron,100a,pdi_T2_nto_pair1_electron.cub
4,pdi,Parent PDI,triplet,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.496971,0.496971,hole,99a,pdi_T2_nto_pair1_hole.cub
7,pdi,Parent PDI,triplet,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,2,0.432008,0.928979,electron,101a,pdi_T2_nto_pair2_electron.cub
6,pdi,Parent PDI,triplet,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,2,0.432008,0.928979,hole,98a,pdi_T2_nto_pair2_hole.cub
9,pdi_terminal_functionalized,Terminal-functionalized PDI,singlet,Singlet,S1,1,2.748,pdi_terminal_functionalized_S1_nto.mwfn,calculations/pdi_terminal_functionalized/multi...,1,0.955154,0.955154,electron,140a,pdi_terminal_functionalized_S1_nto_pair1_elect...
8,pdi_terminal_functionalized,Terminal-functionalized PDI,singlet,Singlet,S1,1,2.748,pdi_terminal_functionalized_S1_nto.mwfn,calculations/pdi_terminal_functionalized/multi...,1,0.955154,0.955154,hole,139a,pdi_terminal_functionalized_S1_nto_pair1_hole.cub


## F. Supporting Information rendering manifest


In [7]:
si_display = (
    si_manifest
    .sort_values(["system_order", "multiplicity_order", "state_label", "pair_index", "orbital_role"])
    .drop(columns=["system_order", "multiplicity_order"], errors="ignore")
    .copy()
)

si_display["excitation_energy_ev"] = si_display["excitation_energy_ev"].round(3)
si_display["pair_weight"] = si_display["pair_weight"].round(6)
si_display["cumulative_weight"] = si_display["cumulative_weight"].round(6)

display(si_display)


,system,system_label,multiplicity,multiplicity_label,state_label,global_state_number,excitation_energy_ev,nto_filename,nto_path,pair_index,pair_weight,cumulative_weight,orbital_role,orbital_label,recommended_cube_basename
1,pdi,Parent PDI,singlet,Singlet,S1,1,2.763,pdi_S1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.955480,0.955480,electron,100a,pdi_S1_nto_pair1_electron.cub
0,pdi,Parent PDI,singlet,Singlet,S1,1,2.763,pdi_S1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.955480,0.955480,hole,99a,pdi_S1_nto_pair1_hole.cub
3,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.908209,0.908209,electron,100a,pdi_T1_nto_pair1_electron.cub
2,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.908209,0.908209,hole,99a,pdi_T1_nto_pair1_hole.cub
5,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,2,0.041624,0.949833,electron,101a,pdi_T1_nto_pair2_electron.cub
4,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,2,0.041624,0.949833,hole,98a,pdi_T1_nto_pair2_hole.cub
7,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,3,0.028193,0.978026,electron,102a,pdi_T1_nto_pair3_electron.cub
6,pdi,Parent PDI,triplet,Triplet,T1,1,1.617,pdi_T1_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,3,0.028193,0.978026,hole,97a,pdi_T1_nto_pair3_hole.cub
9,pdi,Parent PDI,triplet,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.496971,0.496971,electron,100a,pdi_T2_nto_pair1_electron.cub
8,pdi,Parent PDI,triplet,Triplet,T2,2,2.949,pdi_T2_nto.mwfn,calculations/pdi/multiwfn_analysis/excited_sta...,1,0.496971,0.496971,hole,99a,pdi_T2_nto_pair1_hole.cub


## G. Interpretation notes

The parser assigns S/T labels from the excited-state manifold in the ORCA output, then matches each NTO block by ORCA global state number. The `.sN.nto` suffix is therefore treated only as the printed ORCA state-file identifier, not as a multiplicity label.

Use the main manifest for compact figures with cumulative NTO weight ≥ 0.90. Use the SI manifest when a more complete cumulative representation ≥ 0.95 is required. If an output is incomplete or an NTO block came from a folder whose name disagrees with the parsed multiplicity, keep that warning visible when deciding which cubes to render.
